In [ ]:
# Imports shared by the experiment pipeline.
# Optional packages are handled in the same place so setup issues show up early.
import os
import re
import gc
import json
import math
import time
import copy
import random
import warnings
import itertools
from pathlib import Path
from collections import defaultdict, Counter

from IPython import get_ipython
from IPython.display import display

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns
sns.set_context("talk")
sns.set_style("whitegrid")

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, ParameterGrid, StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    log_loss,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

from scipy.special import softmax
from scipy.stats import (
    ttest_rel,
    wilcoxon,
    fisher_exact,
    chi2_contingency,
    pearsonr,
    spearmanr,
    kendalltau,
    binomtest,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models
from torchvision.transforms import InterpolationMode
from torchvision.models import ConvNeXt_Small_Weights

try:
    import imagehash
    HAS_IMAGEHASH = True
except Exception:
    imagehash = None
    HAS_IMAGEHASH = False

try:
    import umap.umap_ as umap
    HAS_UMAP = True
except Exception:
    umap = None
    HAS_UMAP = False

try:
    from transformers import AutoModel, AutoImageProcessor
    HAS_TRANSFORMERS = True
except Exception:
    AutoModel = None
    AutoImageProcessor = None
    HAS_TRANSFORMERS = False

try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except Exception:
    torchinfo_summary = None
    HAS_TORCHINFO = False

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
    HAS_GRADCAM = True
except Exception:
    GradCAM = None
    ClassifierOutputTarget = None
    show_cam_on_image = None
    HAS_GRADCAM = False

try:
    from torchvision.models.detection import (
        fasterrcnn_resnet50_fpn,
        FasterRCNN_ResNet50_FPN_Weights,
    )
    HAS_TV_DETECTION = True
except Exception:
    fasterrcnn_resnet50_fpn = None
    FasterRCNN_ResNet50_FPN_Weights = None
    HAS_TV_DETECTION = False

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("gpu_mem_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("transformers:", HAS_TRANSFORMERS)
print("umap:", HAS_UMAP)
print("torchinfo:", HAS_TORCHINFO)
print("gradcam:", HAS_GRADCAM)
print("imagehash:", HAS_IMAGEHASH)
print("torchvision detection:", HAS_TV_DETECTION)

In [ ]:
# Paths, run flags, and deterministic seed values.
# Keeping the switches in one place makes reruns a lot less messy.
ROOT = Path.cwd()

# Manual overrides
MANIFEST_PATH = None
IMAGES_ROOT = None
LABEL_COL_OVERRIDE = None
PATH_COL_OVERRIDE = None
SPLIT_COL_OVERRIDE = None
# -----------------------------------------------

# Core run flags
RUN_RAW_DINO_EDA = True
RUN_SEARCH = True
RUN_TRAINING = True
RUN_CALIBRATION = True
RUN_ENSEMBLE = True
RUN_EMBEDDINGS = True
RUN_EXPLAINABILITY = True

# Extra flags for the added analysis cells
RUN_CALIBRATION_BEFORE_AFTER = True
RUN_PER_CLASS_COMPARISON = True
RUN_ROBUSTNESS_SUITE = True
RUN_ERROR_TAXONOMY_SUMMARY = True

# Heavier optional bits
RUN_AUGMENTATION_ABLATIONS = False
RUN_PERSON_CENTRIC_BRANCH = False
RUN_PERSON_CENTRIC_HEAD_ONLY_TRAIN = False

SEARCH_SEED = 42
FINAL_SEEDS = [42, 52, 62]

# New split protocol controls
FROZEN_TEST_SEED = 42
CV_N_SPLITS = 5
CV_SPLIT_SEED = 42
FINAL_EPOCH_RULE = "median_best_epoch"
ENSEMBLE_WEIGHT_OBJECTIVE = "log_loss"
ENSEMBLE_WEIGHT_GRID_STEP = 0.01

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

GPU_MEM_GB = None
if torch.cuda.is_available():
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

# keep image size fixed across models for fairness
IMAGE_SIZE = 224

# conservative 8GB policy for ConvNeXt-Small
if GPU_MEM_GB is not None and GPU_MEM_GB < 10:
    BATCH_CANDIDATES = [4, 8, 16]
    GRAD_ACCUM_STEPS = 2
else:
    BATCH_CANDIDATES = [8, 16, 32]
    GRAD_ACCUM_STEPS = 1

# Additional experiment settings
AUGMENT_ABLATION_MODES = ["mixup", "randaugment_light"]
AUGMENT_ABLATION_SEEDS = [42]

ROBUSTNESS_CONDITIONS = ["clean", "grayscale", "blur", "downscale", "center_occlude"]

PERSON_SCORE_THRESHOLD = 0.70
PERSON_PADDING_FRACTION = 0.10

# Main artifacts folder layout
# Use a clearly suffixed root so the CV refactor never overwrites the fixed-split artifacts.
ART = ROOT / "artifacts_v3_cv"
DIRS = {
    "manifests": ART / "manifests",
    "tables": ART / "tables",
    "plots": ART / "plots",
    "logs": ART / "logs",
    "checkpoints": ART / "checkpoints",
    "predictions": ART / "predictions",
    "embeddings": ART / "embeddings",
    "calibration": ART / "calibration",
    "ensemble": ART / "ensemble",
    "explainability": ART / "explainability",
    "configs": ART / "configs",
    "diagrams": ART / "diagrams",
    "ablations": ART / "ablations",
    "robustness": ART / "robustness",
    "person_centric": ART / "person_centric",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

PROJECT_CONFIG = {
    "project_title": "Human Activity Classification from Still Images",
    "supervised_model": "ConvNeXt-Small",
    "ssl_model": "DINOv2-small",
    "ssl_backbone_name": "facebook/dinov2-small",
    "device": str(DEVICE),
    "gpu_mem_gb": None if GPU_MEM_GB is None else round(float(GPU_MEM_GB), 2),
    "image_size": IMAGE_SIZE,
    "search_seed": SEARCH_SEED,
    "final_seeds": FINAL_SEEDS,
    "frozen_test_seed": FROZEN_TEST_SEED,
    "cv_n_splits": CV_N_SPLITS,
    "ensemble_weight_objective": ENSEMBLE_WEIGHT_OBJECTIVE,
    "ensemble_weight_grid_step": ENSEMBLE_WEIGHT_GRID_STEP,
    "cv_split_seed": CV_SPLIT_SEED,
    "final_epoch_rule": FINAL_EPOCH_RULE,
    "batch_candidates": BATCH_CANDIDATES,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "optimizer": "AdamW",
    "scheduler": "ReduceLROnPlateau",
    "search_mode": "balanced",
    "stage_a_max_trials": 16,
    "stage_b_top_k": 4,
    "pilot_epochs": 6,
    "full_epochs": 24,
    "pilot_patience": 2,
    "full_patience": 6,
    "run_calibration_before_after": RUN_CALIBRATION_BEFORE_AFTER,
    "run_per_class_comparison": RUN_PER_CLASS_COMPARISON,
    "run_robustness_suite": RUN_ROBUSTNESS_SUITE,
    "run_error_taxonomy_summary": RUN_ERROR_TAXONOMY_SUMMARY,
    "run_augmentation_ablations": RUN_AUGMENTATION_ABLATIONS,
    "run_person_centric_branch": RUN_PERSON_CENTRIC_BRANCH,
    "split_protocol": "frozen_stratified_test_plus_internal_stratified_cv",
}
with open(DIRS["configs"] / "project_config.json", "w", encoding="utf-8") as f:
    json.dump(PROJECT_CONFIG, f, indent=2)

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEARCH_SEED)
print(json.dumps(PROJECT_CONFIG, indent=2))

In [ ]:
# Standardize the manifest before downstream validation.
# The CV refactor keeps any original split labels for traceability, but the working protocol now
# rebuilds one frozen stratified test split and treats everything else as the non-test CV pool.
PATH_CANDIDATES = ["image_path", "local_path", "path", "file_path", "filepath", "img_path"]
LABEL_CANDIDATES = ["label", "activity", "class", "target", "y"]
SPLIT_CANDIDATES = ["split", "subset", "partition"]

def find_csv_candidates(root: Path):
    csvs = []
    for p in root.rglob("*.csv"):
        p_str = str(p).lower()
        if "results" in p_str and "classification_report" in p_str:
            continue
        csvs.append(p)
    return csvs

def first_match(cols, candidates, override=None):
    if override is not None and override in cols:
        return override
    lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

def score_manifest(df):
    cols = list(df.columns)
    score = 0
    if first_match(cols, PATH_CANDIDATES) is not None:
        score += 3
    if first_match(cols, LABEL_CANDIDATES) is not None:
        score += 3
    if first_match(cols, SPLIT_CANDIDATES) is not None:
        score += 2
    if "image_id" in [c.lower() for c in cols]:
        score += 1
    return score

def discover_manifest():
    if MANIFEST_PATH is not None:
        return Path(MANIFEST_PATH)
    candidates = []
    for p in find_csv_candidates(ROOT):
        try:
            df_head = pd.read_csv(p, nrows=20)
            candidates.append((score_manifest(df_head), p))
        except Exception:
            pass
    candidates = sorted(candidates, key=lambda x: (-x[0], str(x[1])))
    if not candidates or candidates[0][0] < 6:
        raise FileNotFoundError(
            "Could not auto-discover a usable manifest CSV. Please set MANIFEST_PATH manually."
        )
    return candidates[0][1]

def _canonicalize_original_split(series):
    out = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({
            "valid": "val",
            "validation": "val",
            "trainval": "train_val",
            "train_val": "train_val",
        })
    )
    return out

def _target_test_count(df_in):
    default_target = int(round(len(df_in) * 0.15))
    default_target = max(default_target, int(df_in["label"].nunique()))
    default_target = min(default_target, len(df_in) - int(df_in["label"].nunique()))

    if "original_split" in df_in.columns:
        existing_test = int((df_in["original_split"] == "test").sum())
        if existing_test > 0:
            default_target = existing_test
    return int(default_target)

def _build_frozen_protocol_split(df_in, seed, target_test_count):
    df_work = df_in.reset_index(drop=True).copy()
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=int(target_test_count),
        random_state=int(seed),
    )
    indices = np.arange(len(df_work))
    pool_idx, test_idx = next(splitter.split(indices, df_work["label"]))

    df_work["split"] = "non_test_pool"
    df_work.loc[test_idx, "split"] = "test"
    df_work["protocol_split"] = df_work["split"]
    df_work["frozen_test_flag"] = df_work["split"] == "test"
    df_work["non_test_pool_flag"] = df_work["split"] == "non_test_pool"
    df_work["protocol_row_id"] = np.arange(len(df_work))
    df_work["frozen_test_seed"] = int(seed)
    df_work["frozen_test_target_count"] = int(target_test_count)
    return df_work

manifest_path = discover_manifest()
raw_df = pd.read_csv(manifest_path)
print("Using manifest:", manifest_path)
print("Rows:", len(raw_df))
print("Columns:", list(raw_df.columns))

path_col = first_match(raw_df.columns, PATH_CANDIDATES, PATH_COL_OVERRIDE)
label_col = first_match(raw_df.columns, LABEL_CANDIDATES, LABEL_COL_OVERRIDE)
split_col = first_match(raw_df.columns, SPLIT_CANDIDATES, SPLIT_COL_OVERRIDE)

if path_col is None or label_col is None:
    raise ValueError("Manifest must contain an image-path column and a label column.")

df = raw_df.copy()
# A portable manifest may retain both the public split and its explicit provenance.
# Verify the two views agree before reducing them to the loader's canonical schema.
if split_col is not None and split_col != "original_split" and "original_split" in df.columns:
    declared_original_split = _canonicalize_original_split(df["original_split"])
    declared_split = _canonicalize_original_split(df[split_col])
    if not declared_original_split.equals(declared_split):
        raise ValueError("split and original_split disagree in the input manifest.")
    df = df.drop(columns=["original_split"])

rename_map = {path_col: "image_path", label_col: "label"}
if split_col is not None:
    rename_map[split_col] = "original_split"
df = df.rename(columns=rename_map)

if "image_id" not in df.columns:
    df["image_id"] = np.arange(len(df))

df["label"] = df["label"].astype(str).str.strip()
df["image_path"] = df["image_path"].astype(str)

if IMAGES_ROOT is not None:
    df["image_path"] = df["image_path"].apply(
        lambda p: str((Path(IMAGES_ROOT) / p).resolve()) if not Path(p).is_absolute() else p
    )

if "original_split" in df.columns:
    df["original_split"] = _canonicalize_original_split(df["original_split"])
else:
    df["original_split"] = "unspecified"

target_test_count = _target_test_count(df)
df = _build_frozen_protocol_split(
    df_in=df,
    seed=FROZEN_TEST_SEED,
    target_test_count=target_test_count,
)

df["image_path"] = df["image_path"].apply(lambda x: str(Path(x)))
df["exists"] = df["image_path"].apply(lambda x: Path(x).exists())

if not df["exists"].all():
    missing_count = int((~df["exists"]).sum())
    print(f"Warning: {missing_count} image paths do not exist.")

protocol_counts = (
    df.groupby(["split", "label"])
      .size()
      .rename("count")
      .reset_index()
      .sort_values(["split", "label"])
      .reset_index(drop=True)
)
protocol_counts.to_csv(DIRS["tables"] / "T00_protocol_split_counts.csv", index=False)

protocol_summary = pd.DataFrame(
    [{
        "manifest_path": str(manifest_path),
        "n_rows": int(len(df)),
        "n_unique_image_ids": int(df["image_id"].nunique()),
        "n_classes": int(df["label"].nunique()),
        "original_splits_present": ", ".join(sorted(df["original_split"].astype(str).unique())),
        "working_protocol_splits": ", ".join(sorted(df["split"].astype(str).unique())),
        "frozen_test_seed": int(FROZEN_TEST_SEED),
        "frozen_test_target_count": int(target_test_count),
    }]
)
protocol_summary.to_csv(DIRS["tables"] / "T00_protocol_summary.csv", index=False)

df.to_csv(DIRS["manifests"] / "standardized_manifest.csv", index=False)
df.to_csv(DIRS["manifests"] / "standardized_manifest_cv_protocol.csv", index=False)

display(protocol_summary)
display(protocol_counts)
df.head()

In [ ]:
# Check image metadata, decodability, duplicates, and split leakage.
# It is better to catch those issues now than after training starts.
def safe_image_open(path):
    try:
        with Image.open(path) as im:
            im = im.convert("RGB")
            return im.copy()
    except Exception:
        return None

def compute_image_metadata(df_in: pd.DataFrame):
    rows = []
    for _, row in tqdm(df_in.iterrows(), total=len(df_in), desc="Metadata scan"):
        p = Path(row["image_path"])
        im = safe_image_open(p)

        if im is None:
            rows.append({
                "image_id": row["image_id"],
                "width_meta": np.nan,
                "height_meta": np.nan,
                "aspect_ratio_meta": np.nan,
                "phash_meta": None,
                "integrity_status_meta": "corrupt_or_unreadable",
            })
            continue

        width, height = im.size
        aspect_ratio = width / height if height > 0 else np.nan

        if HAS_IMAGEHASH:
            ph = str(imagehash.phash(im))
        else:
            ph = None

        rows.append({
            "image_id": row["image_id"],
            "width_meta": width,
            "height_meta": height,
            "aspect_ratio_meta": aspect_ratio,
            "phash_meta": ph,
            "integrity_status_meta": "ok",
        })

    return pd.DataFrame(rows)



# remove stale columns from previous bad runs if present
stale_cols = [
    "width", "height", "aspect_ratio", "phash", "integrity_status",
    "width_x", "width_y", "height_x", "height_y",
    "aspect_ratio_x", "aspect_ratio_y",
    "phash_x", "phash_y",
    "integrity_status_x", "integrity_status_y",
]
drop_now = [c for c in stale_cols if c in df.columns]
if drop_now:
    df = df.drop(columns=drop_now)

meta_extra = compute_image_metadata(df[["image_id", "image_path"]].drop_duplicates())
df = df.merge(meta_extra, on="image_id", how="left")

# rename back to the canonical names
df = df.rename(columns={
    "width_meta": "width",
    "height_meta": "height",
    "aspect_ratio_meta": "aspect_ratio",
    "phash_meta": "phash",
    "integrity_status_meta": "integrity_status",
})

# duplicate-group analysis
if "phash" in df.columns and df["phash"].notna().any():
    dup_groups = (
        df.dropna(subset=["phash"])
          .groupby("phash")
          .agg(
              n=("image_id", "count"),
              splits=("split", lambda x: sorted(set(x))),
              labels=("label", lambda x: sorted(set(x))),
              paths=("image_path", lambda x: list(x)[:5]),
          )
          .reset_index()
    )
    dup_groups = dup_groups[dup_groups["n"] > 1].sort_values("n", ascending=False)
else:
    dup_groups = pd.DataFrame(columns=["phash", "n", "splits", "labels", "paths"])

cross_split_dup_count = 0
if len(dup_groups) > 0:
    cross_split_dup_count = dup_groups["splits"].apply(lambda x: len(x) > 1).sum()

integrity_summary = (
    df.groupby(["split", "integrity_status"])
      .size()
      .rename("count")
      .reset_index()
)

dup_summary = pd.DataFrame({
    "duplicate_groups_total": [len(dup_groups)],
    "duplicate_groups_cross_split": [int(cross_split_dup_count)],
    "total_rows": [len(df)],
    "unique_image_ids": [df["image_id"].nunique()],
})

Path(DIRS["manifests"]).mkdir(parents=True, exist_ok=True)
Path(DIRS["tables"]).mkdir(parents=True, exist_ok=True)

df.to_csv(Path(DIRS["manifests"]) / "standardized_manifest_with_meta.csv", index=False)
dup_groups.to_csv(Path(DIRS["tables"]) / "duplicate_groups.csv", index=False)
integrity_summary.to_csv(Path(DIRS["tables"]) / "integrity_summary.csv", index=False)
dup_summary.to_csv(Path(DIRS["tables"]) / "duplicate_summary.csv", index=False)

print("Columns after clean metadata merge:")
print(list(df.columns))
print()
display(dup_summary)
display(integrity_summary.head())

In [ ]:
# Transform, dataset, loader, and cross-validation definitions.
# The development pool uses internal CV; the original test split remains fixed.
def in_notebook():
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        return shell == "ZMQInteractiveShell"
    except Exception:
        return False

IS_WINDOWS = os.name == "nt"
IN_NOTEBOOK = in_notebook()

SAFE_NUM_WORKERS = 0

class AddGaussianNoise:
    def __init__(self, std=0.01):
        self.std = std

    def __call__(self, tensor):
        return torch.clamp(tensor + torch.randn_like(tensor) * self.std, 0.0, 1.0)

def get_model_norms():
    return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_train_transform(strength="moderate"):
    mean, std = get_model_norms()

    if strength == "mild":
        return transforms.Compose([
            transforms.RandomResizedCrop(
                IMAGE_SIZE,
                scale=(0.75, 1.00),
                ratio=(0.80, 1.25),
                interpolation=InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.RandomAffine(
                    degrees=8,
                    translate=(0.04, 0.04),
                    scale=(0.95, 1.05),
                    shear=2,
                    interpolation=InterpolationMode.BICUBIC,
                )
            ], p=0.35),
            transforms.ColorJitter(
                brightness=0.18,
                contrast=0.18,
                saturation=0.12,
                hue=0.03,
            ),
            transforms.ToTensor(),
            transforms.RandomApply([AddGaussianNoise(std=0.01)], p=0.10),
            transforms.RandomErasing(
                p=0.15,
                scale=(0.02, 0.08),
                ratio=(0.3, 3.0),
                value="random",
            ),
            transforms.Normalize(mean=mean, std=std),
        ])

    if strength == "moderate":
        return transforms.Compose([
            transforms.RandomResizedCrop(
                IMAGE_SIZE,
                scale=(0.60, 1.00),
                ratio=(0.75, 1.33),
                interpolation=InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.RandomAffine(
                    degrees=10,
                    translate=(0.06, 0.06),
                    scale=(0.90, 1.10),
                    shear=4,
                    interpolation=InterpolationMode.BICUBIC,
                )
            ], p=0.50),
            transforms.ColorJitter(
                brightness=0.25,
                contrast=0.25,
                saturation=0.20,
                hue=0.04,
            ),
            transforms.ToTensor(),
            transforms.RandomApply([AddGaussianNoise(std=0.015)], p=0.12),
            transforms.RandomErasing(
                p=0.25,
                scale=(0.02, 0.12),
                ratio=(0.3, 3.3),
                value="random",
            ),
            transforms.Normalize(mean=mean, std=std),
        ])

    raise ValueError(f"Unknown augmentation strength: {strength}")

def build_eval_transform():
    mean, std = get_model_norms()
    return transforms.Compose([
        transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

label_encoder = LabelEncoder()
label_encoder.fit(df["label"])
label_map = {i: cls for i, cls in enumerate(label_encoder.classes_)}

with open(DIRS["configs"] / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

class ActivityDataset(Dataset):
    def __init__(self, df_, label_encoder_, transform):
        self.df = df_.reset_index(drop=True).copy()
        self.le = label_encoder_
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_image_open(row["image_path"])
        if img is None:
            raise ValueError(f"Unreadable image: {row['image_path']}")
        x = self.transform(img)
        y = int(self.le.transform([row["label"]])[0])
        return {
            "pixel_values": x,
            "label": y,
            "image_id": row["image_id"],
            "image_path": row["image_path"],
            "label_text": row["label"],
            "cv_row_id": row.get("cv_row_id", np.nan),
            "split": row.get("split", None),
        }

def make_loader(df_, transform, batch_size, shuffle, seed):
    ds = ActivityDataset(df_, label_encoder, transform)

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=SAFE_NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
        generator=generator,
    )

non_test_pool_df = (
    df[(df["split"] == "non_test_pool") & (df["integrity_status"] == "ok")]
    .reset_index(drop=True)
    .copy()
)
non_test_pool_df["cv_row_id"] = np.arange(len(non_test_pool_df))

test_df = (
    df[(df["split"] == "test") & (df["integrity_status"] == "ok")]
    .reset_index(drop=True)
    .copy()
)
test_df["cv_row_id"] = np.nan

def get_cv_folds(pool_df_=None, n_splits=CV_N_SPLITS, seed=CV_SPLIT_SEED):
    pool_df_ = non_test_pool_df if pool_df_ is None else pool_df_
    pool_df_ = pool_df_.reset_index(drop=True).copy()

    if "cv_row_id" not in pool_df_.columns:
        pool_df_["cv_row_id"] = np.arange(len(pool_df_))

    skf = StratifiedKFold(
        n_splits=int(n_splits),
        shuffle=True,
        random_state=int(seed),
    )

    folds = []
    assignment_rows = []
    for fold_id, (train_idx, val_idx) in enumerate(skf.split(pool_df_, pool_df_["label"]), start=1):
        fold_train = pool_df_.iloc[train_idx].copy().reset_index(drop=True)
        fold_val = pool_df_.iloc[val_idx].copy().reset_index(drop=True)
        folds.append({
            "fold_id": int(fold_id),
            "train_df": fold_train,
            "val_df": fold_val,
            "train_indices": train_idx,
            "val_indices": val_idx,
        })
        tmp = fold_val[["image_id", "image_path", "label", "cv_row_id"]].copy()
        tmp["fold_id"] = int(fold_id)
        assignment_rows.append(tmp)

    assignment_df = pd.concat(assignment_rows, axis=0).sort_values("cv_row_id").reset_index(drop=True)
    return folds, assignment_df

CV_FOLDS, cv_fold_assignment_df = get_cv_folds(non_test_pool_df, n_splits=CV_N_SPLITS, seed=CV_SPLIT_SEED)
cv_fold_assignment_df.to_csv(DIRS["tables"] / "T03c_cv_fold_assignment.csv", index=False)

protocol_split_df = (
    pd.DataFrame(
        {
            "split": ["non_test_pool", "test"],
            "count": [len(non_test_pool_df), len(test_df)],
            "percent": [100.0 * len(non_test_pool_df) / max(1, len(df)), 100.0 * len(test_df) / max(1, len(df))],
        }
    )
)
protocol_split_df.to_csv(DIRS["tables"] / "T03d_protocol_split_summary.csv", index=False)

cv_fold_size_df = (
    cv_fold_assignment_df.groupby(["fold_id", "label"])
    .size()
    .rename("count")
    .reset_index()
)
cv_fold_size_df.to_csv(DIRS["tables"] / "T03e_cv_fold_class_counts.csv", index=False)

print("IS_WINDOWS:", IS_WINDOWS)
print("IN_NOTEBOOK:", IN_NOTEBOOK)
print("SAFE_NUM_WORKERS:", SAFE_NUM_WORKERS)
print("Non-test pool / Test:", len(non_test_pool_df), len(test_df))
print("CV folds:", CV_N_SPLITS, "with seed", CV_SPLIT_SEED)
print("Classes:", list(label_encoder.classes_))
display(protocol_split_df)
display(cv_fold_size_df)

In [ ]:
# Model builders and architecture summaries.
# Saved configuration files keep architecture metadata aligned with selected runs.
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import ConvNeXt_Small_Weights

try:
    from transformers import AutoModel
except Exception:
    AutoModel = None

Path(DIRS["diagrams"]).mkdir(parents=True, exist_ok=True)
Path(DIRS["tables"]).mkdir(parents=True, exist_ok=True)

def _infer_n_classes():
    if "label_map" in globals():
        try:
            return len(label_map)
        except Exception:
            pass
    if "label_encoder" in globals():
        try:
            return len(label_encoder.classes_)
        except Exception:
            pass
    if "df" in globals() and "label" in df.columns:
        try:
            return int(pd.Series(df["label"]).nunique())
        except Exception:
            pass
    return 3

N_CLASSES = _infer_n_classes()


def build_convnext_small(num_classes=N_CLASSES):
    model = models.convnext_small(weights=ConvNeXt_Small_Weights.IMAGENET1K_V1)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

def freeze_convnext(model, strategy="head_only"):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    if strategy == "head_only":
        return model

    if strategy == "last_stage":
        
        # ONLY unfreezing the last downsample + stage 4
        for name, p in model.named_parameters():
            if name.startswith("features.6") or name.startswith("features.7"):
                p.requires_grad = True
        return model

    if strategy == "full_backbone":
        for p in model.parameters():
            p.requires_grad = True
        return model

    raise ValueError(strategy)

class Dinov2Classifier(nn.Module):
    def __init__(self, backbone_name="facebook/dinov2-small", num_classes=N_CLASSES, dropout=0.0):
        super().__init__()
        if (AutoModel is None) or (not HAS_TRANSFORMERS):
            raise ImportError("transformers / AutoModel is required for DINOv2-small.")
        self.backbone_name = backbone_name
        self.backbone = AutoModel.from_pretrained(backbone_name)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, pixel_values, return_features=False, output_attentions=False):
        out = self.backbone(pixel_values=pixel_values, output_attentions=output_attentions)
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            feats = out.pooler_output
        else:
            feats = out.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(feats))

        if return_features and output_attentions:
            return logits, feats, out.attentions
        if return_features:
            return logits, feats
        return logits


def get_dino_block_names(model_or_backbone):
    target = model_or_backbone.backbone if hasattr(model_or_backbone, "backbone") else model_or_backbone
    block_names = []
    for name, _ in target.named_parameters():
        m = re.search(r"(encoder\.layer\.\d+)", name)
        if m:
            block_names.append(m.group(1))
        m2 = re.search(r"(vit\.encoder\.layer\.\d+)", name)
        if m2:
            block_names.append(m2.group(1))
    block_names = sorted(set(block_names), key=lambda s: int(re.findall(r"\d+", s)[-1]))
    return block_names

def _matches_dino_block(full_name, block_name):
    return (
        full_name.startswith(block_name)
        or full_name.startswith(f"backbone.{block_name}")
        or f".{block_name}." in full_name
        or full_name == block_name
        or full_name == f"backbone.{block_name}"
    )

def freeze_dino(model, strategy="probe_only", top_n_blocks=4):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    if strategy == "probe_only":
        return model

    block_names = get_dino_block_names(model)
    if strategy == "top_blocks":
        chosen = set(block_names[-top_n_blocks:])
        for name, p in model.backbone.named_parameters():
            if any(_matches_dino_block(name, block) for block in chosen):
                p.requires_grad = True
        for name, p in model.backbone.named_parameters():
            if "layernorm" in name.lower() or "norm" in name.lower():
                p.requires_grad = True
        return model

    if strategy == "full_backbone":
        for p in model.parameters():
            p.requires_grad = True
        return model

    raise ValueError(strategy)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return total, trainable, frozen

def _fmt_m(n):
    return f"{n / 1e6:.2f}M"

def _count_prefixed_params(model, prefixes):
    total = 0
    trainable = 0
    for name, p in model.named_parameters():
        if any(name.startswith(pref) for pref in prefixes):
            total += p.numel()
            if p.requires_grad:
                trainable += p.numel()
    return total, trainable

def _count_grouped_params(model, group_fn):
    total = 0
    trainable = 0
    for name, p in model.named_parameters():
        if group_fn(name):
            total += p.numel()
            if p.requires_grad:
                trainable += p.numel()
    return total, trainable

def _trainability_label(total, trainable):
    if total == 0:
        return "n/a"
    if trainable == 0:
        return "Frozen"
    if trainable == total:
        return "Trainable"
    return f"Partial ({_fmt_m(trainable)}/{_fmt_m(total)})"


conv_cfg_path = Path(DIRS["configs"]) / "best_convnext_small_cfg.json"
dino_cfg_path = Path(DIRS["configs"]) / "best_dinov2_small_cfg.json"

conv_best_cfg_local = None
if conv_cfg_path.exists():
    try:
        with open(conv_cfg_path, "r", encoding="utf-8") as f:
            conv_best_cfg_local = json.load(f)
    except Exception:
        conv_best_cfg_local = None
if conv_best_cfg_local is None:
    conv_best_cfg_local = globals().get("best_conv_cfg", {"unfreeze_strategy": "full_backbone"})

dino_best_cfg_local = None
if dino_cfg_path.exists():
    try:
        with open(dino_cfg_path, "r", encoding="utf-8") as f:
            dino_best_cfg_local = json.load(f)
    except Exception:
        dino_best_cfg_local = None
if dino_best_cfg_local is None:
    dino_best_cfg_local = globals().get("best_dino_cfg", {"unfreeze_strategy": "top_blocks", "top_n_blocks": 4})

conv_best_strategy = str(conv_best_cfg_local.get("unfreeze_strategy", "full_backbone"))
if conv_best_strategy not in {"head_only", "last_stage", "full_backbone"}:
    conv_best_strategy = "full_backbone"

dino_best_strategy = str(dino_best_cfg_local.get("unfreeze_strategy", "top_blocks"))
if dino_best_strategy not in {"probe_only", "top_blocks", "full_backbone"}:
    dino_best_strategy = "top_blocks"

dino_top_n = int(dino_best_cfg_local.get("top_n_blocks", 4))


convnext_probe = freeze_convnext(build_convnext_small(), conv_best_strategy)
dino_probe = freeze_dino(
    Dinov2Classifier(backbone_name=PROJECT_CONFIG["ssl_backbone_name"]),
    dino_best_strategy,
    top_n_blocks=dino_top_n,
)

summary_rows = []
for name, model in [("convnext_small", convnext_probe), ("dinov2_small", dino_probe)]:
    total, trainable, frozen = count_parameters(model)
    summary_rows.append({
        "model": name,
        "total_params": total,
        "trainable_params": trainable,
        "frozen_params": frozen,
        "input_resolution": IMAGE_SIZE,
        "n_classes": N_CLASSES,
    })

arch_df = pd.DataFrame(summary_rows)
arch_df.to_csv(Path(DIRS["tables"]) / "T05_architecture_parameter_summary.csv", index=False)
display(arch_df)

if "HAS_TORCHINFO" in globals() and HAS_TORCHINFO:
    try:
        print("ConvNeXt-Small summary")
        print(torchinfo_summary(convnext_probe, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), depth=3, verbose=0))
    except Exception as e:
        print("torchinfo convnext failed:", e)

    try:
        print("DINOv2-small summary")
        print(torchinfo_summary(dino_probe, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), depth=3, verbose=0))
    except Exception as e:
        print("torchinfo dino failed:", e)


conv_s0 = IMAGE_SIZE // 4
conv_s1 = conv_s0
conv_s2 = conv_s1 // 2
conv_s3 = conv_s2 // 2
conv_s4 = conv_s3 // 2

conv_stage_1_blocks = len(convnext_probe.features[1])
conv_stage_2_blocks = len(convnext_probe.features[3])
conv_stage_3_blocks = len(convnext_probe.features[5])
conv_stage_4_blocks = len(convnext_probe.features[7])

conv_stage_specs = [
    {
        "module": "stem",
        "prefixes": ["features.0"],
        "output_shape": f"{conv_s0}x{conv_s0}x96",
        "core_operation": "4x4 stride-4 conv + LayerNorm",
        "hook": "",
    },
    {
        "module": "stage_1",
        "prefixes": ["features.1"],
        "output_shape": f"{conv_s0}x{conv_s0}x96",
        "core_operation": f"{conv_stage_1_blocks} ConvNeXt blocks",
        "hook": "",
    },
    {
        "module": "stage_2",
        "prefixes": ["features.2", "features.3"],
        "output_shape": f"{conv_s2}x{conv_s2}x192",
        "core_operation": f"downsample x2 + {conv_stage_2_blocks} ConvNeXt blocks",
        "hook": "",
    },
    {
        "module": "stage_3",
        "prefixes": ["features.4", "features.5"],
        "output_shape": f"{conv_s3}x{conv_s3}x384",
        "core_operation": f"downsample x2 + {conv_stage_3_blocks} ConvNeXt blocks",
        "hook": "",
    },
    {
        "module": "stage_4",
        "prefixes": ["features.6", "features.7"],
        "output_shape": f"{conv_s4}x{conv_s4}x768",
        "core_operation": f"downsample x2 + {conv_stage_4_blocks} ConvNeXt blocks",
        "hook": "Convolutional attribution source",
    },
    {
        "module": "gap_embedding",
        "prefixes": ["classifier.0", "classifier.1"],
        "output_shape": "1x768",
        "core_operation": "global average pooling + norm",
        "hook": "Embedding extraction",
    },
    {
        "module": "classifier_head",
        "prefixes": ["classifier.2"],
        "output_shape": f"{N_CLASSES} logits",
        "core_operation": "linear 768 -> class logits",
        "hook": "",
    },
]

conv_stage_rows = []
for spec in conv_stage_specs:
    total, trainable = _count_prefixed_params(convnext_probe, spec["prefixes"])
    conv_stage_rows.append({
        "model": "convnext_small",
        "module": spec["module"],
        "output_shape": spec["output_shape"],
        "core_operation": spec["core_operation"],
        "trainability": _trainability_label(total, trainable),
        "params_total": total,
        "params_trainable": trainable,
        "params_total_m": _fmt_m(total),
        "params_trainable_m": _fmt_m(trainable),
        "hook": spec["hook"],
    })

conv_stage_df = pd.DataFrame(conv_stage_rows)
conv_stage_df.to_csv(Path(DIRS["tables"]) / "T05b_convnext_stagewise_summary.csv", index=False)
display(conv_stage_df)

dino_hidden = int(getattr(dino_probe.backbone.config, "hidden_size", 384))
dino_patch = int(getattr(dino_probe.backbone.config, "patch_size", 14))
dino_heads = int(getattr(dino_probe.backbone.config, "num_attention_heads", 6))
dino_blocks = get_dino_block_names(dino_probe)
dino_total_blocks = len(dino_blocks)

if dino_best_strategy == "top_blocks":
    top_n = min(max(dino_top_n, 0), dino_total_blocks)
elif dino_best_strategy == "probe_only":
    top_n = 0
else:
    top_n = dino_total_blocks

lower_blocks = dino_blocks[:-top_n] if top_n > 0 else dino_blocks
top_blocks = dino_blocks[-top_n:] if top_n > 0 else []

token_grid = IMAGE_SIZE // dino_patch if (IMAGE_SIZE % dino_patch == 0) else None
if token_grid is not None:
    token_text = f"{token_grid * token_grid + 1} tokens x {dino_hidden}"
else:
    token_text = f"tokens x {dino_hidden}"


def _dino_group(name):
    lname = name.lower()

    if name.startswith("classifier"):
        return "classifier_head"

    if name.startswith("backbone.embeddings") or name.startswith("backbone.cls_token") or name.startswith("backbone.mask_token"):
        return "patch_and_embeddings"

    if any(_matches_dino_block(name, b) for b in top_blocks):
        return "top_blocks"

    if any(_matches_dino_block(name, b) for b in lower_blocks):
        return "lower_blocks"

    if name.startswith("backbone.layernorm") or "layernorm" in lname or ".norm" in lname:
        return "backbone_norms"

    if name.startswith("backbone"):
        return "patch_and_embeddings"

    return "other"

dino_group_rows = []
for group_name, op, out_shape, hook in [
    ("patch_and_embeddings", f"patch size {dino_patch} + pos/cls embeddings", token_text, ""),
    ("lower_blocks", f"first {len(lower_blocks)} transformer blocks", token_text, ""),
    ("top_blocks", f"top {len(top_blocks)} transformer blocks", token_text, "Attention and gradient source"),
    ("backbone_norms", "backbone norms", token_text, ""),
    ("classifier_head", f"linear {dino_hidden} -> class logits", f"{N_CLASSES} logits", ""),
]:
    total, trainable = _count_grouped_params(dino_probe, lambda n, g=group_name: _dino_group(n) == g)
    dino_group_rows.append({
        "model": "dinov2_small",
        "module": group_name,
        "output_shape": out_shape,
        "core_operation": op,
        "trainability": _trainability_label(total, trainable),
        "params_total": total,
        "params_trainable": trainable,
        "params_total_m": _fmt_m(total),
        "params_trainable_m": _fmt_m(trainable),
        "hook": hook,
    })

dino_stage_df = pd.DataFrame(dino_group_rows)
dino_stage_df.to_csv(Path(DIRS["tables"]) / "T05c_dinov2_stagewise_summary.csv", index=False)
display(dino_stage_df)

conv_total_params_expected = int(arch_df.loc[arch_df["model"] == "convnext_small", "total_params"].iloc[0])
dino_total_params_expected = int(arch_df.loc[arch_df["model"] == "dinov2_small", "total_params"].iloc[0])

conv_total_params_stagewise = int(conv_stage_df["params_total"].sum())
dino_total_params_stagewise = int(dino_stage_df["params_total"].sum())

if conv_total_params_stagewise != conv_total_params_expected:
    print(f"Warning: ConvNeXt stagewise params ({conv_total_params_stagewise}) do not sum to model total ({conv_total_params_expected}).")
if dino_total_params_stagewise != dino_total_params_expected:
    print(f"Warning: DINOv2 stagewise params ({dino_total_params_stagewise}) do not sum to model total ({dino_total_params_expected}).")

model_card_df = pd.DataFrame([
    {
        "model": "convnext_small",
        "architecture_family": "hierarchical ConvNet",
        "input": f"{IMAGE_SIZE}x{IMAGE_SIZE} RGB",
        "feature_path": "4x4 stride-4 stem -> 4 stages -> GAP -> 768-d embedding",
        "output": f"{N_CLASSES} logits -> softmax probabilities",
        "target_encoding": "class index / one-hot equivalent",
        "loss": "softmax cross-entropy",
        "selected_final_regime": conv_best_strategy,
        "embedding_source": "pooled 1x768 feature after GAP + norm",
        "explainability_source": "OOF-selected attribution from Stage 4 feature maps",
    },
    {
        "model": "dinov2_small",
        "architecture_family": "ViT-style self-supervised encoder",
        "input": f"{IMAGE_SIZE}x{IMAGE_SIZE} RGB",
        "feature_path": f"patchify ({dino_patch}x{dino_patch}) -> {token_text} -> CLS -> linear head",
        "output": f"{N_CLASSES} logits -> softmax probabilities",
        "target_encoding": "class index / one-hot equivalent",
        "loss": "softmax cross-entropy",
        "selected_final_regime": f"{dino_best_strategy} (top_n_blocks={top_n})",
        "embedding_source": f"CLS feature after backbone norm ({dino_hidden}-d)",
        "explainability_source": "OOF-selected class-specific transformer attribution",
    },
])
model_card_df.to_csv(Path(DIRS["tables"]) / "T05d_model_card_summary.csv", index=False)
display(model_card_df)


COLOR = {
    "input": "#DBEAFE",
    "preprocess": "#E0F2FE",
    "embed": "#EDE9FE",
    "frozen": "#F3F4F6",
    "trainable": "#FEF3C7",
    "feature": "#DCFCE7",
    "head": "#FECACA",
    "output": "#FDE68A",
    "hook": "#F3E8FF",
    "panel": "#F9FAFB",
    "border": "#374151",
    "text": "#111827",
    "muted": "#6B7280",
}

def _setup_axis(ax, xlim=(0, 215), ylim=(0, 76)):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axis("off")

def _draw_info_box(ax, x, y, w, h, title, shape_line, op_line, status_line, extra_line, fc, badge_text=""):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.4,
        edgecolor=COLOR["border"],
        facecolor=fc,
    )
    ax.add_patch(patch)

    if badge_text:
        badge_w = 5.2
        badge = FancyBboxPatch(
            (x + w - badge_w - 0.8, y + h - 3.0),
            badge_w,
            1.7,
            boxstyle="round,pad=0.02,rounding_size=0.05",
            linewidth=0.9,
            edgecolor=COLOR["border"],
            facecolor="white",
        )
        ax.add_patch(badge)
        ax.text(
            x + w - badge_w / 2 - 0.8,
            y + h - 2.15,
            badge_text,
            ha="center",
            va="center",
            fontsize=7.0,
            color=COLOR["text"],
            weight="bold",
        )

    ax.text(x + w / 2, y + h * 0.84, title, ha="center", va="center", fontsize=10.4, weight="bold", color=COLOR["text"])
    ax.text(x + w / 2, y + h * 0.66, shape_line, ha="center", va="center", fontsize=9.1, weight="bold", color=COLOR["text"])
    ax.text(x + w / 2, y + h * 0.45, op_line, ha="center", va="center", fontsize=7.7, color=COLOR["muted"])
    ax.text(x + w / 2, y + h * 0.24, status_line, ha="center", va="center", fontsize=7.8, color=COLOR["text"])
    ax.text(x + w / 2, y + h * 0.08, extra_line, ha="center", va="center", fontsize=7.2, color=COLOR["muted"])
    return patch

def _arrow(ax, start_xy, end_xy, dashed=False, lw=1.8):
    a = FancyArrowPatch(
        start_xy,
        end_xy,
        arrowstyle="->",
        mutation_scale=14,
        linewidth=lw,
        linestyle="--" if dashed else "-",
        color=COLOR["border"],
    )
    ax.add_patch(a)

def _small_label(ax, x, y, text):
    ax.text(x, y, text, fontsize=8.0, color=COLOR["muted"], ha="center", va="center")

def _group_bracket(ax, x1, x2, y, label):
    ax.plot([x1, x2], [y, y], color=COLOR["border"], linewidth=1.2)
    ax.plot([x1, x1], [y, y - 1.4], color=COLOR["border"], linewidth=1.2)
    ax.plot([x2, x2], [y, y - 1.4], color=COLOR["border"], linewidth=1.2)
    ax.text((x1 + x2) / 2, y + 1.0, label, ha="center", va="bottom", fontsize=9.0, color=COLOR["text"], weight="bold")

def _draw_side_panel(ax, x, y, w, h, title, lines):
    panel = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04,rounding_size=0.08",
        linewidth=1.3,
        edgecolor=COLOR["border"],
        facecolor=COLOR["panel"],
    )
    ax.add_patch(panel)
    ax.text(x + 1.5, y + h - 1.8, title, ha="left", va="center", fontsize=10.5, weight="bold", color=COLOR["text"])

    yy = y + h - 4.2
    for line in lines:
        ax.text(x + 1.5, yy, line, ha="left", va="top", fontsize=8.3, color=COLOR["text"])
        yy -= 2.6

def _legend(ax, x0, y0):
    items = [
        ("Input / preprocessing", COLOR["input"]),
        ("Frozen backbone", COLOR["frozen"]),
        ("Trainable blocks", COLOR["trainable"]),
        ("Feature / embedding", COLOR["feature"]),
        ("Classifier / output", COLOR["head"]),
        ("Analysis hook", COLOR["hook"]),
    ]
    cur_x = x0
    for label, fc in items:
        rect = FancyBboxPatch(
            (cur_x, y0),
            3.0,
            1.2,
            boxstyle="round,pad=0.02,rounding_size=0.04",
            linewidth=1.0,
            edgecolor=COLOR["border"],
            facecolor=fc,
        )
        ax.add_patch(rect)
        ax.text(cur_x + 3.8, y0 + 0.6, label, va="center", fontsize=7.8, color=COLOR["text"])
        cur_x += 17.0

def _save_figure(fig, stem):
    png_path = Path(DIRS["diagrams"]) / f"{stem}.png"
    svg_path = Path(DIRS["diagrams"]) / f"{stem}.svg"
    fig.savefig(png_path, dpi=220, bbox_inches="tight", facecolor="white")
    fig.savefig(svg_path, bbox_inches="tight", facecolor="white")
    print(f"Saved: {png_path}")
    print(f"Saved: {svg_path}")

def _trainability_fill(label, role="backbone"):
    if role == "input":
        return COLOR["input"]
    if role == "preprocess":
        return COLOR["preprocess"]
    if role == "embed":
        return COLOR["embed"]
    if role == "feature":
        return COLOR["feature"]
    if role == "head":
        return COLOR["head"]
    if role == "output":
        return COLOR["output"]
    if role == "hook":
        return COLOR["hook"]

    if isinstance(label, str):
        if label.startswith("Trainable"):
            return COLOR["trainable"]
        if label.startswith("Frozen"):
            return COLOR["frozen"]
        if label.startswith("Partial"):
            return COLOR["embed"]
    return COLOR["panel"]

def _trainability_badge_text(label):
    if not isinstance(label, str):
        return ""
    if label.startswith("Trainable"):
        return "Trainable"
    if label.startswith("Frozen"):
        return "Frozen"
    if label.startswith("Partial"):
        return "Partial"
    return ""

def _convnext_regime_text(strategy):
    if strategy == "head_only":
        return "head-only transfer"
    if strategy == "last_stage":
        return "last-stage fine-tuning"
    if strategy == "full_backbone":
        return "full-backbone fine-tuning"
    return str(strategy)

def _dino_regime_text(strategy, top_n_blocks):
    if strategy == "probe_only":
        return "linear probe"
    if strategy == "top_blocks":
        return f"top-{top_n_blocks}-block fine-tuning + backbone norms"
    if strategy == "full_backbone":
        return "full-backbone fine-tuning"
    return str(strategy)

conv_lookup = {row["module"]: row for _, row in conv_stage_df.iterrows()}
dino_lookup = {row["module"]: row for _, row in dino_stage_df.iterrows()}

def draw_convnext_diagram():
    fig, ax = plt.subplots(figsize=(24, 8.7), facecolor="white")
    _setup_axis(ax)

    conv_regime_text = _convnext_regime_text(conv_best_strategy)

    ax.text(2, 72.0, "F07c ConvNeXt-Small architecture diagram", fontsize=16, weight="bold", color=COLOR["text"])
    ax.text(2, 69.8, f"Selected final fine-tuning regime: {conv_regime_text}", fontsize=9.5, color=COLOR["muted"])

    y = 23
    h = 22
    w = 17

    boxes = [
        {
            "x": 2, "role": "input", "trainability": "",
            "title": "Input",
            "shape": "224x224x3",
            "op": "RGB still image",
            "status": "Data only",
            "extra": "Task: 3-class activity classification",
        },
        {
            "x": 21, "role": "preprocess", "trainability": "",
            "title": "Preprocess",
            "shape": "224x224x3",
            "op": "train: crop/flip/affine\njitter/noise/erase",
            "status": "eval: resize + center crop",
            "extra": "Normalization: ImageNet statistics",
        },
        {
            "x": 40, "role": "backbone", "trainability": conv_lookup["stem"]["trainability"],
            "title": "Stem",
            "shape": conv_lookup["stem"]["output_shape"],
            "op": "4x4 stride-4 conv + LayerNorm",
            "status": f"Status: {conv_lookup['stem']['trainability']}",
            "extra": f"Params: {conv_lookup['stem']['params_total_m']}",
        },
        {
            "x": 59, "role": "backbone", "trainability": conv_lookup["stage_1"]["trainability"],
            "title": "Stage 1",
            "shape": conv_lookup["stage_1"]["output_shape"],
            "op": f"{conv_stage_1_blocks} ConvNeXt blocks",
            "status": f"Status: {conv_lookup['stage_1']['trainability']}",
            "extra": f"Params: {conv_lookup['stage_1']['params_total_m']}",
        },
        {
            "x": 78, "role": "backbone", "trainability": conv_lookup["stage_2"]["trainability"],
            "title": "Stage 2",
            "shape": conv_lookup["stage_2"]["output_shape"],
            "op": f"downsample x2 + {conv_stage_2_blocks} ConvNeXt blocks",
            "status": f"Status: {conv_lookup['stage_2']['trainability']}",
            "extra": f"Params: {conv_lookup['stage_2']['params_total_m']}",
        },
        {
            "x": 97, "role": "backbone", "trainability": conv_lookup["stage_3"]["trainability"],
            "title": "Stage 3",
            "shape": conv_lookup["stage_3"]["output_shape"],
            "op": f"downsample x2 + {conv_stage_3_blocks} ConvNeXt blocks",
            "status": f"Status: {conv_lookup['stage_3']['trainability']}",
            "extra": f"Params: {conv_lookup['stage_3']['params_total_m']}",
        },
        {
            "x": 116, "role": "backbone", "trainability": conv_lookup["stage_4"]["trainability"],
            "title": "Stage 4",
            "shape": conv_lookup["stage_4"]["output_shape"],
            "op": f"downsample x2 + {conv_stage_4_blocks} ConvNeXt blocks",
            "status": f"Status: {conv_lookup['stage_4']['trainability']}",
            "extra": f"Hook: {conv_lookup['stage_4']['hook']} | Params: {conv_lookup['stage_4']['params_total_m']}",
        },
        {
            "x": 135, "role": "feature", "trainability": conv_lookup["gap_embedding"]["trainability"],
            "title": "GAP / Embedding",
            "shape": conv_lookup["gap_embedding"]["output_shape"],
            "op": "global average pooling + norm",
            "status": f"Status: {conv_lookup['gap_embedding']['trainability']}",
            "extra": "Embedding source: pooled 1x768",
        },
        {
            "x": 154, "role": "head", "trainability": conv_lookup["classifier_head"]["trainability"],
            "title": "Linear Head",
            "shape": conv_lookup["classifier_head"]["output_shape"],
            "op": "linear 768 -> class logits",
            "status": f"Status: {conv_lookup['classifier_head']['trainability']}",
            "extra": f"Params: {conv_lookup['classifier_head']['params_total_m']}",
        },
        {
            "x": 173, "role": "output", "trainability": "",
            "title": "Softmax",
            "shape": f"{N_CLASSES} probs",
            "op": "probability normalization",
            "status": "Inference only",
            "extra": "Prediction rule: argmax",
        },
    ]

    for b in boxes:
        fc = _trainability_fill(b["trainability"], role=b["role"])
        badge = _trainability_badge_text(b["trainability"]) if b["role"] == "backbone" else ""
        _draw_info_box(
            ax,
            b["x"], y, w, h,
            b["title"], b["shape"], b["op"], b["status"], b["extra"],
            fc,
            badge_text=badge,
        )

    for i in range(len(boxes) - 1):
        _arrow(ax, (boxes[i]["x"] + w, y + h / 2), (boxes[i + 1]["x"], y + h / 2))

    _small_label(ax, 87.5, 34.3, "downsample x2")
    _small_label(ax, 106.5, 34.3, "downsample x2")
    _small_label(ax, 125.5, 34.3, "downsample x2")

    _group_bracket(ax, 2, 38, 50.5, "Preprocessing")
    _group_bracket(ax, 40, 133, 50.5, "Backbone")
    _group_bracket(ax, 135, 152, 50.5, "Feature extraction")
    _group_bracket(ax, 154, 190, 50.5, "Classifier / output")
    _group_bracket(ax, 115.5, 154.5, 65.0, "Analysis hooks")

    _draw_info_box(
        ax, 115.5, 53.3, 18, 9.0,
        "Attribution source", "Stage 4 feature map", "last conv stage used", "OOF-selected faithfulness audit", "",
        COLOR["hook"]
    )
    _draw_info_box(
        ax, 136.5, 53.3, 25, 9.0,
        "Embedding hook", "Pooled 1x768", "feature extraction point", "PCA / UMAP / retrieval", "",
        COLOR["hook"]
    )
    _arrow(ax, (124.5, y + h), (124.5, 53.2), dashed=True)
    _arrow(ax, (143.5, y + h), (145.5, 53.2), dashed=True)

    total_params = int(arch_df.loc[arch_df["model"] == "convnext_small", "total_params"].iloc[0])
    trainable_params = int(arch_df.loc[arch_df["model"] == "convnext_small", "trainable_params"].iloc[0])

    _draw_side_panel(
        ax,
        193, 17.5, 22, 46.0,
        "Model card",
        [
            f"Total params: {_fmt_m(total_params)}",
            f"Trainable params: {_fmt_m(trainable_params)}",
            f"Selected regime: {conv_regime_text}",
            "Backbone family: hierarchical ConvNet",
            "Block pattern: 7x7 depthwise conv + pointwise MLP + residual",
            "Target: class index / one-hot equivalent",
            "Loss: softmax cross-entropy",
            "Embedding source: pooled 1x768 feature",
            "Explainability: OOF-selected CAM audit",
        ],
    )

    _legend(ax, 2, 2.0)
    ax.text(
        2, 11.5,
        f"Training path: head warm-up -> {conv_regime_text}. Input = 224x224 RGB. Output = {N_CLASSES} logits -> softmax probabilities.",
        fontsize=8.4,
        color=COLOR["muted"],
    )

    plt.tight_layout()
    _save_figure(fig, "F07c_convnext_small_architecture_diagram")
    plt.show()

def draw_dino_diagram():
    fig, ax = plt.subplots(figsize=(24, 8.8), facecolor="white")
    _setup_axis(ax)

    dino_regime_text = _dino_regime_text(dino_best_strategy, top_n)

    ax.text(2, 72.0, "F08c DINOv2-Small architecture diagram", fontsize=16, weight="bold", color=COLOR["text"])
    ax.text(2, 69.8, f"Selected final fine-tuning regime: {dino_regime_text}", fontsize=9.5, color=COLOR["muted"])

    y = 23
    h = 22
    w = 18

    patch_shape = token_text
    lower_desc = f"first {len(lower_blocks)} transformer blocks"
    top_desc = f"top {len(top_blocks)} transformer blocks"

    norm_total, norm_trainable = _count_grouped_params(dino_probe, lambda n: _dino_group(n) == "backbone_norms")
    patch_total, patch_trainable = _count_grouped_params(dino_probe, lambda n: _dino_group(n) == "patch_and_embeddings")
    lower_total, lower_trainable = _count_grouped_params(dino_probe, lambda n: _dino_group(n) == "lower_blocks")
    top_total, top_trainable = _count_grouped_params(dino_probe, lambda n: _dino_group(n) == "top_blocks")
    head_total, head_trainable = _count_grouped_params(dino_probe, lambda n: _dino_group(n) == "classifier_head")

    boxes = [
        {
            "x": 2, "role": "input", "trainability": "",
            "title": "Input",
            "shape": "224x224x3",
            "op": "RGB still image",
            "status": "Data only",
            "extra": "Task: 3-class activity classification",
        },
        {
            "x": 22, "role": "preprocess", "trainability": "",
            "title": "Preprocess",
            "shape": "224x224x3",
            "op": "train: crop/flip/affine\njitter/noise/erase",
            "status": "eval: resize + center crop",
            "extra": "Normalization: ImageNet statistics",
        },
        {
            "x": 42, "role": "embed", "trainability": _trainability_label(patch_total, patch_trainable),
            "title": "Patch / Pos Embed",
            "shape": patch_shape,
            "op": f"patch size {dino_patch} + CLS + positional embed",
            "status": f"Status: {_trainability_label(patch_total, patch_trainable)}",
            "extra": f"Params: {_fmt_m(patch_total)}",
        },
        {
            "x": 62, "role": "backbone", "trainability": _trainability_label(lower_total, lower_trainable),
            "title": "Lower Blocks",
            "shape": patch_shape,
            "op": lower_desc,
            "status": f"Status: {_trainability_label(lower_total, lower_trainable)}",
            "extra": f"Params: {_fmt_m(lower_total)}",
        },
        {
            "x": 82, "role": "backbone", "trainability": _trainability_label(top_total, top_trainable),
            "title": "Top Blocks",
            "shape": patch_shape,
            "op": top_desc if len(top_blocks) > 0 else "no fine-tuned blocks",
            "status": f"Status: {_trainability_label(top_total, top_trainable)}",
            "extra": f"Hook: attention + gradients | Params: {_fmt_m(top_total)}",
        },
        {
            "x": 102, "role": "feature", "trainability": _trainability_label(norm_total, norm_trainable),
            "title": "CLS Feature",
            "shape": f"1x{dino_hidden}",
            "op": "backbone norm -> CLS feature",
            "status": f"Norm status: {_trainability_label(norm_total, norm_trainable)}",
            "extra": f"Embedding source: CLS ({dino_hidden}-d)",
        },
        {
            "x": 122, "role": "head", "trainability": _trainability_label(head_total, head_trainable),
            "title": "Linear Head",
            "shape": f"{N_CLASSES} logits",
            "op": f"linear {dino_hidden} -> class logits",
            "status": f"Status: {_trainability_label(head_total, head_trainable)}",
            "extra": f"Params: {_fmt_m(head_total)}",
        },
        {
            "x": 142, "role": "output", "trainability": "",
            "title": "Softmax",
            "shape": f"{N_CLASSES} probs",
            "op": "probability normalization",
            "status": "Inference only",
            "extra": "Prediction rule: argmax",
        },
    ]

    for b in boxes:
        fc = _trainability_fill(b["trainability"], role=b["role"])
        badge = _trainability_badge_text(b["trainability"]) if b["role"] in {"embed", "backbone", "feature"} else ""
        _draw_info_box(
            ax,
            b["x"], y, w, h,
            b["title"], b["shape"], b["op"], b["status"], b["extra"],
            fc,
            badge_text=badge,
        )

    for i in range(len(boxes) - 1):
        _arrow(ax, (boxes[i]["x"] + w, y + h / 2), (boxes[i + 1]["x"], y + h / 2))

    _group_bracket(ax, 2, 40, 50.5, "Preprocessing")
    _group_bracket(ax, 42, 100, 50.5, "Backbone")
    _group_bracket(ax, 102, 120, 50.5, "Feature extraction")
    _group_bracket(ax, 122, 160, 50.5, "Classifier / output")
    _group_bracket(ax, 82.5, 121.5, 65.0, "Analysis hooks")

    _draw_info_box(
        ax, 82.5, 53.3, 18, 9.0,
        "Attribution source", "Encoder attentions", "target-conditioned tracing", "OOF-selected faithfulness audit", "",
        COLOR["hook"]
    )
    _draw_info_box(
        ax, 103.5, 53.3, 18, 9.0,
        "Embedding hook", f"CLS 1x{dino_hidden}", "feature extraction point", "PCA / UMAP / retrieval", "",
        COLOR["hook"]
    )
    _arrow(ax, (91.0, y + h), (91.0, 53.2), dashed=True)
    _arrow(ax, (111.0, y + h), (112.0, 53.2), dashed=True)

    total_params = int(arch_df.loc[arch_df["model"] == "dinov2_small", "total_params"].iloc[0])
    trainable_params = int(arch_df.loc[arch_df["model"] == "dinov2_small", "trainable_params"].iloc[0])

    _draw_side_panel(
        ax,
        163, 17.5, 28.0, 46.0,
        "Model card",
        [
            f"Patch size: {dino_patch}",
            f"Hidden size: {dino_hidden}",
            f"Transformer blocks: {dino_total_blocks}",
            f"Selected regime: {dino_regime_text}",
            f"Total params: {_fmt_m(total_params)}",
            f"Trainable params: {_fmt_m(trainable_params)}",
            "Backbone family: ViT-style self-supervised encoder",
            "Block pattern: MHSA + MLP + residual",
            "Target: class index / one-hot equivalent",
            "Loss: softmax cross-entropy",
            "Explainability: class-specific gradient-attention rollout",
        ],
    )

    _legend(ax, 2, 2.0)
    ax.text(
        2, 11.5,
        f"Training path: linear probe -> {dino_regime_text}. Input = 224x224 RGB. Output = {N_CLASSES} logits -> softmax probabilities.",
        fontsize=8.4,
        color=COLOR["muted"],
    )

    plt.tight_layout()
    _save_figure(fig, "F08c_dinov2_small_architecture_diagram")
    plt.show()

draw_convnext_diagram()
draw_dino_diagram()

In [ ]:
# Label encoding and shared metric helpers.
# These utilities get reused by training, calibration, and final evaluation.
SAFE_NUM_WORKERS = 0

def make_loader(df_, transform, batch_size, shuffle, seed):
    ds = ActivityDataset(df_, label_encoder, transform)

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=SAFE_NUM_WORKERS,   
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
        generator=generator,
    )

def multiclass_brier_score(y_true, probs, num_classes):
    y_oh = label_binarize(y_true, classes=np.arange(num_classes))
    if y_oh.shape[1] != num_classes:
        tmp = np.zeros((len(y_true), num_classes))
        tmp[np.arange(len(y_true)), y_true] = 1
        y_oh = tmp
    return np.mean(np.sum((probs - y_oh) ** 2, axis=1))

def ece_mce(y_true, probs, n_bins=15):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    bin_rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        if mask.sum() == 0:
            bin_rows.append({"bin": i, "count": 0, "acc": np.nan, "conf": np.nan})
            continue

        acc = correct[mask].mean()
        c = conf[mask].mean()
        gap = abs(acc - c)

        ece += (mask.sum() / len(y_true)) * gap
        mce = max(mce, gap)

        bin_rows.append({
            "bin": i,
            "count": int(mask.sum()),
            "acc": float(acc),
            "conf": float(c),
        })

    return ece, mce, pd.DataFrame(bin_rows)

def basic_metrics(y_true, probs):
    preds = probs.argmax(axis=1)

    p, r, f, s = precision_recall_fscore_support(
        y_true, preds, average=None, zero_division=0
    )
    macro_p, macro_r, macro_f, _ = precision_recall_fscore_support(
        y_true, preds, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f, _ = precision_recall_fscore_support(
        y_true, preds, average="weighted", zero_division=0
    )

    metrics = {
        "accuracy": accuracy_score(y_true, preds),
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f,
        "weighted_f1": weighted_f,
        "balanced_accuracy": balanced_accuracy_score(y_true, preds),
        "log_loss": log_loss(y_true, probs, labels=np.arange(probs.shape[1])),
        "brier_score": multiclass_brier_score(y_true, probs, probs.shape[1]),
    }

    try:
        y_bin = label_binarize(y_true, classes=np.arange(probs.shape[1]))
        metrics["macro_ovr_roc_auc"] = roc_auc_score(
            y_bin, probs, average="macro", multi_class="ovr"
        )
        metrics["macro_ovr_pr_auc"] = average_precision_score(
            y_bin, probs, average="macro"
        )
    except Exception:
        metrics["macro_ovr_roc_auc"] = np.nan
        metrics["macro_ovr_pr_auc"] = np.nan

    ece, mce, cal_df = ece_mce(y_true, probs, n_bins=15)
    metrics["ece"] = ece
    metrics["mce"] = mce

    per_class = pd.DataFrame({
        "class_id": np.arange(len(label_encoder.classes_)),
        "class_name": label_encoder.classes_,
        "precision": p,
        "recall": r,
        "f1": f,
        "support": s,
    })

    return metrics, per_class, cal_df


try:
    _test_loader = make_loader(
        non_test_pool_df,
        build_train_transform("moderate"),
        batch_size=min(2, len(non_test_pool_df)),
        shuffle=False,
        seed=SEARCH_SEED,
    )
    _batch = next(iter(_test_loader))
    print("make_loader sanity check passed")
    print("SAFE_NUM_WORKERS =", SAFE_NUM_WORKERS)
    print("batch tensor shape:", _batch["pixel_values"].shape)
except Exception as e:
    print("make_loader sanity check FAILED")
    raise

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temp = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self):
        return torch.exp(self.log_temp)

    def forward(self, logits):
        return logits / self.temperature.clamp(min=1e-6)

def fit_temperature_scaler(val_logits, val_labels, max_iter=1000, lr=0.01):
    scaler = TemperatureScaler().to(DEVICE)
    logits_t = torch.tensor(val_logits, dtype=torch.float32, device=DEVICE)
    labels_t = torch.tensor(val_labels, dtype=torch.long, device=DEVICE)

    optim = torch.optim.Adam([scaler.log_temp], lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_loss = float("inf")
    for _ in range(max_iter):
        optim.zero_grad()
        loss = criterion(scaler(logits_t), labels_t)
        loss.backward()
        optim.step()
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = copy.deepcopy(scaler.state_dict())

    scaler.load_state_dict(best_state)
    return scaler, best_loss

In [ ]:
# Training, validation, and artifact serialization primitives.
# The point of this cell is to keep the update logic in one place.
class EarlyStopper:
    def __init__(self, patience=5, mode="max"):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.num_bad = 0
        self.best_state = None
        self.best_epoch = None

    def step(self, metric, model, epoch):
        improved = False

        if self.best is None:
            improved = True
        elif self.mode == "max" and metric > self.best:
            improved = True
        elif self.mode == "min" and metric < self.best:
            improved = True

        if improved:
            self.best = metric
            self.num_bad = 0
            self.best_state = copy.deepcopy(model.state_dict())
            self.best_epoch = epoch
            return False
        else:
            self.num_bad += 1
            return self.num_bad >= self.patience

def make_optimizer_and_scheduler(model, lr_head, lr_backbone=None, weight_decay=1e-4):
    head_params = []
    backbone_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "classifier" in name:
            head_params.append(p)
        else:
            backbone_params.append(p)

    param_groups = []
    if len(head_params) > 0:
        param_groups.append({"params": head_params, "lr": lr_head})

    if len(backbone_params) > 0:
        param_groups.append({
            "params": backbone_params,
            "lr": lr_backbone if lr_backbone is not None else lr_head
        })

    if len(param_groups) == 0:
        raise ValueError("No trainable parameters were found for optimizer construction.")

    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1
    )

    return optimizer, scheduler

def loss_fn_builder(label_smoothing=0.0):
    return nn.CrossEntropyLoss(label_smoothing=label_smoothing)

def get_current_lrs(optimizer):
    return [pg["lr"] for pg in optimizer.param_groups]

def compute_grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return total ** 0.5

def train_one_epoch(model, loader, optimizer, criterion, scaler=None, grad_accum_steps=1):
    model.train()
    losses = []
    grad_norms = []

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        x = batch["pixel_values"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y) / grad_accum_steps

        if USE_AMP:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        should_step = ((step + 1) % grad_accum_steps == 0) or ((step + 1) == len(loader))
        if should_step:
            if USE_AMP:
                scaler.unscale_(optimizer)
            grad_norms.append(compute_grad_norm(model))

            if USE_AMP:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item() * grad_accum_steps)

    return float(np.mean(losses)), float(np.mean(grad_norms)) if grad_norms else np.nan

@torch.no_grad()
def evaluate_model(model, loader, criterion, return_features=False, return_attentions=False):
    model.eval()
    losses = []
    all_logits, all_probs, all_labels, all_preds = [], [], [], []
    all_paths, all_ids, all_features = [], [], []
    all_attentions = []

    for batch in loader:
        x = batch["pixel_values"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            if return_features and return_attentions:
                logits, feats, attn = model(x, return_features=True, output_attentions=True)
            elif return_features:
                logits, feats = model(x, return_features=True)
                attn = None
            else:
                logits = model(x)
                feats = None
                attn = None

            loss = criterion(logits, y)

        probs = torch.softmax(logits, dim=1)

        losses.append(loss.item())
        all_logits.append(logits.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())
        all_preds.append(probs.argmax(dim=1).detach().cpu().numpy())
        all_paths.extend(batch["image_path"])
        all_ids.extend(batch["image_id"])

        if feats is not None:
            all_features.append(feats.detach().cpu().numpy())
        if attn is not None:
            # store only last batch attention if caller explicitly wants full attention processing later
            all_attentions.append([a.detach().cpu() for a in attn])

    logits = np.concatenate(all_logits, axis=0)
    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    preds = np.concatenate(all_preds, axis=0)
    features = np.concatenate(all_features, axis=0) if len(all_features) else None

    metrics, per_class_df, cal_df = basic_metrics(labels, probs)

    output = {
        "loss": float(np.mean(losses)),
        "labels": labels,
        "preds": preds,
        "logits": logits,
        "probs": probs,
        "paths": all_paths,
        "image_ids": all_ids,
        "features": features,
        "metrics": metrics,
        "per_class_df": per_class_df,
        "calibration_df": cal_df,
        "attentions": all_attentions,
    }
    return output

def save_prediction_bundle(out_dir, split_name, eval_output):
    out_dir.mkdir(parents=True, exist_ok=True)

    if isinstance(eval_output.get("pred_df", None), pd.DataFrame):
        pred_df = eval_output["pred_df"].copy()
    else:
        pred_df = pd.DataFrame({
            "image_id": eval_output["image_ids"],
            "image_path": eval_output["paths"],
            "y_true": eval_output["labels"],
            "y_pred": eval_output["preds"],
            "confidence": eval_output["probs"].max(axis=1),
        })

    pred_df.to_csv(out_dir / f"{split_name}_predictions.csv", index=False)

    np.save(out_dir / f"{split_name}_logits.npy", eval_output["logits"])
    np.save(out_dir / f"{split_name}_probs.npy", eval_output["probs"])
    np.save(out_dir / f"{split_name}_labels.npy", eval_output["labels"])
    if eval_output["features"] is not None:
        np.save(out_dir / f"{split_name}_features.npy", eval_output["features"])

    pd.DataFrame([eval_output["metrics"]]).to_csv(out_dir / f"{split_name}_metrics.csv", index=False)
    eval_output["per_class_df"].to_csv(out_dir / f"{split_name}_classification_report.csv", index=False)
    eval_output["calibration_df"].to_csv(out_dir / f"{split_name}_calibration_bins.csv", index=False)

In [ ]:
# Cross-validation and fixed-epoch full-pool training orchestration.
# That includes the best checkpoint, the per-epoch history, and the saved metadata that calibration and comparison cells read back in.
def _json_safe_obj(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe_obj(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe_obj(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(float(obj)) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def build_model_and_strategy(model_kind, cfg):
    if model_kind == "convnext_small":
        model = build_convnext_small()
        model = freeze_convnext(model, cfg["unfreeze_strategy"])
        return model

    if model_kind == "dinov2_small":
        model = Dinov2Classifier(backbone_name=PROJECT_CONFIG["ssl_backbone_name"])
        if cfg["unfreeze_strategy"] == "probe_only":
            model = freeze_dino(model, "probe_only")
        elif cfg["unfreeze_strategy"] == "top_blocks":
            model = freeze_dino(model, "top_blocks", top_n_blocks=cfg["top_n_blocks"])
        elif cfg["unfreeze_strategy"] == "full_backbone":
            model = freeze_dino(model, "full_backbone")
        else:
            raise ValueError(cfg["unfreeze_strategy"])
        return model

    raise ValueError(model_kind)


def run_single_train(
    model_kind,
    cfg,
    train_df_,
    val_df_,
    seed,
    epochs,
    patience,
    run_name,
):
    seed_everything(seed)

    train_tf = build_train_transform(cfg["augmentation_strength"])
    eval_tf = build_eval_transform()

    train_loader = make_loader(train_df_, train_tf, cfg["batch_size"], shuffle=True, seed=seed)
    val_loader = make_loader(val_df_, eval_tf, cfg["batch_size"], shuffle=False, seed=seed)

    model = build_model_and_strategy(model_kind, cfg).to(DEVICE)
    total_params = int(sum(p.numel() for p in model.parameters()))
    trainable_params = int(sum(p.numel() for p in model.parameters() if p.requires_grad))

    criterion = loss_fn_builder(label_smoothing=cfg.get("label_smoothing", 0.0))
    optimizer, scheduler = make_optimizer_and_scheduler(
        model,
        lr_head=cfg["head_lr"],
        lr_backbone=cfg.get("backbone_lr", None),
        weight_decay=cfg["weight_decay"],
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    stopper = EarlyStopper(patience=patience, mode="max")

    history = []
    ckpt_dir = DIRS["checkpoints"] / model_kind / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_checkpoint_path = ckpt_dir / "best_checkpoint.pt"

    with open(ckpt_dir / "run_config.json", "w", encoding="utf-8") as f:
        json.dump(_json_safe_obj(cfg), f, indent=2)

    for epoch in range(1, epochs + 1):
        if torch.cuda.is_available():
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass

        t0 = time.perf_counter()

        train_loss, grad_norm = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            grad_accum_steps=GRAD_ACCUM_STEPS,
        )

        val_out = evaluate_model(model, val_loader, criterion, return_features=False, return_attentions=False)
        val_metrics = val_out["metrics"]

        scheduler.step(val_metrics["macro_f1"])

        epoch_seconds = float(time.perf_counter() - t0)
        peak_memory_mb = float(torch.cuda.max_memory_allocated() / 1024**2) if torch.cuda.is_available() else np.nan

        should_stop = stopper.step(val_metrics["macro_f1"], model, epoch)
        is_best_epoch = bool(stopper.best_epoch == epoch)

        epoch_row = {
            "run_name": run_name,
            "model_kind": model_kind,
            "seed": int(seed),
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_out["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "grad_norm": grad_norm,
            "lr_group_0": optimizer.param_groups[0]["lr"],
            "epoch_seconds": epoch_seconds,
            "peak_memory_mb": peak_memory_mb,
            "is_best_epoch": is_best_epoch,
            "checkpoint_dir": str(ckpt_dir),
            "best_checkpoint_path": str(best_checkpoint_path),
            "total_params": total_params,
            "trainable_params": trainable_params,
        }
        if len(optimizer.param_groups) > 1:
            epoch_row["lr_group_1"] = optimizer.param_groups[1]["lr"]

        history.append(epoch_row)

        if should_stop:
            break

    model.load_state_dict(stopper.best_state)

    history_df = pd.DataFrame(history)
    history_df.to_csv(ckpt_dir / "training_history.csv", index=False)

    summary_payload = {
        "model_kind": model_kind,
        "run_name": run_name,
        "seed": int(seed),
        "best_epoch": int(stopper.best_epoch),
        "best_val_macro_f1": float(stopper.best),
        "epochs_ran": int(history_df["epoch"].max()),
        "mean_epoch_seconds": float(history_df["epoch_seconds"].mean()) if "epoch_seconds" in history_df.columns else np.nan,
        "peak_memory_mb": float(history_df["peak_memory_mb"].max()) if "peak_memory_mb" in history_df.columns else np.nan,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "checkpoint_dir": str(ckpt_dir),
        "best_checkpoint_path": str(best_checkpoint_path),
    }
    with open(ckpt_dir / "run_summary.json", "w", encoding="utf-8") as f:
        json.dump(_json_safe_obj(summary_payload), f, indent=2)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "cfg": cfg,
            "seed": seed,
            "model_kind": model_kind,
            "best_epoch": stopper.best_epoch,
            "best_val_macro_f1": stopper.best,
            "label_map": label_map,
            "total_params": total_params,
            "trainable_params": trainable_params,
        },
        best_checkpoint_path,
    )

    return {
        "model": model,
        "history_df": history_df,
        "best_epoch": stopper.best_epoch,
        "best_val_macro_f1": stopper.best,
        "ckpt_dir": ckpt_dir,
        "best_checkpoint_path": best_checkpoint_path,
        "summary": summary_payload,
    }

# --- CV helpers added for the frozen-test + internal-CV protocol ---

def _ensure_cv_pool_df(pool_df_):
    pool_df_ = pool_df_.copy().reset_index(drop=True)
    if "cv_row_id" not in pool_df_.columns:
        pool_df_["cv_row_id"] = np.arange(len(pool_df_), dtype=int)
    if "image_id" in pool_df_.columns:
        pool_df_["image_id"] = pool_df_["image_id"].astype(str)
    return pool_df_

def get_cv_folds_for_seed(pool_df_, n_splits=CV_N_SPLITS, split_seed=CV_SPLIT_SEED):
    pool_df_ = _ensure_cv_pool_df(pool_df_)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=split_seed)
    X_dummy = np.zeros(len(pool_df_), dtype=int)
    y = pool_df_["label"].astype(str).values
    return [(tr_idx, va_idx) for tr_idx, va_idx in skf.split(X_dummy, y)]

def _build_pred_df_for_eval(eval_output, source_df, seed, fold_id=None, split_role="oof"):
    source_df = source_df.reset_index(drop=True).copy()
    pred_df = pd.DataFrame({
        "cv_row_id": source_df["cv_row_id"].values if "cv_row_id" in source_df.columns else np.arange(len(source_df)),
        "image_id": source_df["image_id"].astype(str).values if "image_id" in source_df.columns else np.arange(len(source_df)).astype(str),
        "image_path": source_df["image_path"].values,
        "label": source_df["label"].values if "label" in source_df.columns else None,
        "seed": int(seed),
        "split_role": split_role,
    })
    if fold_id is not None:
        pred_df["fold_id"] = int(fold_id)
    pred_df["y_true"] = eval_output["labels"]
    pred_df["y_pred"] = eval_output["preds"]
    pred_df["confidence"] = eval_output["probs"].max(axis=1)
    return pred_df

def _concat_sorted_array_df(frames, prefix):
    if not frames:
        return None
    out = pd.concat(frames, axis=0, ignore_index=True)
    out = out.sort_values("cv_row_id").reset_index(drop=True)
    value_cols = [c for c in out.columns if c.startswith(prefix)]
    return out, out[value_cols].to_numpy()

def _aggregate_fold_histories(history_frames, fold_summaries):
    if len(history_frames) == 0:
        return pd.DataFrame()

    hist_df = pd.concat(history_frames, axis=0, ignore_index=True)
    numeric_cols = [
        c for c in [
            "train_loss",
            "val_loss",
            "val_accuracy",
            "val_macro_f1",
            "val_weighted_f1",
            "val_balanced_accuracy",
            "grad_norm",
            "lr_group_0",
            "lr_group_1",
            "epoch_seconds",
            "peak_memory_mb",
        ]
        if c in hist_df.columns
    ]
    agg_map = {c: ["mean", "std"] for c in numeric_cols}
    agg_df = hist_df.groupby("epoch").agg(agg_map).reset_index()
    agg_df.columns = [
        "_".join([str(x) for x in col if str(x) != ""]).strip("_")
        for col in agg_df.columns.to_flat_index()
    ]
    agg_df["run_name"] = hist_df["run_name"].iloc[0].rsplit("_fold", 1)[0]
    agg_df["model_kind"] = hist_df["model_kind"].iloc[0]
    agg_df["seed"] = int(hist_df["seed"].iloc[0])
    agg_df["cv_n_splits"] = int(hist_df["fold_id"].nunique())
    if len(fold_summaries) > 0:
        fold_best_epochs = [int(x["best_epoch"]) for x in fold_summaries]
        agg_df["cv_best_epoch_mean"] = float(np.mean(fold_best_epochs))
        agg_df["cv_best_epoch_median"] = float(np.median(fold_best_epochs))
    return agg_df

def _build_oof_output(pool_df_, seed, pred_frames, logits_frames, probs_frames, features_frames=None):
    pred_df = pd.concat(pred_frames, axis=0, ignore_index=True).sort_values("cv_row_id").reset_index(drop=True)

    logits_df, logits = _concat_sorted_array_df(logits_frames, "logit_")
    probs_df, probs = _concat_sorted_array_df(probs_frames, "prob_")
    if logits_df is None or probs_df is None:
        raise RuntimeError("Missing OOF logits/probabilities while assembling the pooled CV bundle.")

    labels = pred_df["y_true"].to_numpy(dtype=int)
    preds = pred_df["y_pred"].to_numpy(dtype=int)
    paths = pred_df["image_path"].tolist()
    image_ids = pred_df["image_id"].astype(str).tolist()

    features = None
    if features_frames:
        feat_out, feat_arr = _concat_sorted_array_df(features_frames, "feat_")
        features = feat_arr

    metrics, per_class_df, cal_df = basic_metrics(labels, probs)

    return {
        "loss": float(np.nan),
        "labels": labels,
        "preds": preds,
        "logits": logits,
        "probs": probs,
        "paths": paths,
        "image_ids": image_ids,
        "features": features,
        "metrics": metrics,
        "per_class_df": per_class_df,
        "calibration_df": cal_df,
        "attentions": [],
        "pred_df": pred_df,
    }

def derive_final_epoch_from_folds(fold_summaries, rule=FINAL_EPOCH_RULE):
    best_epochs = [int(fs["best_epoch"]) for fs in fold_summaries]
    if len(best_epochs) == 0:
        raise ValueError("Cannot derive a final epoch count without fold summaries.")

    if rule == "median_best_epoch":
        return int(max(1, round(float(np.median(best_epochs)))))
    if rule == "mean_best_epoch":
        return int(max(1, round(float(np.mean(best_epochs)))))
    raise ValueError(f"Unknown FINAL_EPOCH_RULE: {rule}")

def run_cv_for_cfg(
    model_kind,
    cfg,
    pool_df_,
    seed,
    epochs,
    patience,
    run_name_prefix,
    n_splits=CV_N_SPLITS,
    split_seed=CV_SPLIT_SEED,
    save_fold_predictions=False,
):
    pool_df_ = _ensure_cv_pool_df(pool_df_)
    folds = get_cv_folds_for_seed(pool_df_, n_splits=n_splits, split_seed=split_seed)

    fold_summaries = []
    fold_metric_rows = []
    history_frames = []
    pred_frames, logits_frames, probs_frames, features_frames = [], [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(folds, 1):
        fold_train_df = pool_df_.iloc[train_idx].copy().reset_index(drop=True)
        fold_val_df = pool_df_.iloc[val_idx].copy().reset_index(drop=True)

        fold_run_name = f"{run_name_prefix}_fold{fold_idx:02d}"
        out = run_single_train(
            model_kind=model_kind,
            cfg=cfg,
            train_df_=fold_train_df,
            val_df_=fold_val_df,
            seed=seed,
            epochs=epochs,
            patience=patience,
            run_name=fold_run_name,
        )

        model = out["model"]
        criterion = loss_fn_builder(label_smoothing=cfg.get("label_smoothing", 0.0))
        eval_tf = build_eval_transform()
        fold_loader = make_loader(fold_val_df, eval_tf, int(cfg["batch_size"]), shuffle=False, seed=seed)
        fold_out = evaluate_model(model, fold_loader, criterion, return_features=False)

        pred_df = _build_pred_df_for_eval(fold_out, fold_val_df, seed=seed, fold_id=fold_idx, split_role="oof_validation")
        fold_out["pred_df"] = pred_df

        pred_frames.append(pred_df)
        logits_fold = pd.DataFrame(fold_out["logits"], columns=[f"logit_{i}" for i in range(fold_out["logits"].shape[1])])
        logits_fold["cv_row_id"] = pred_df["cv_row_id"].values
        logits_frames.append(logits_fold)

        probs_fold = pd.DataFrame(fold_out["probs"], columns=[f"prob_{i}" for i in range(fold_out["probs"].shape[1])])
        probs_fold["cv_row_id"] = pred_df["cv_row_id"].values
        probs_frames.append(probs_fold)

        if fold_out.get("features", None) is not None:
            feat_fold = pd.DataFrame(fold_out["features"], columns=[f"feat_{i}" for i in range(fold_out["features"].shape[1])])
            feat_fold["cv_row_id"] = pred_df["cv_row_id"].values
            features_frames.append(feat_fold)

        hist_df = out["history_df"].copy()
        hist_df["fold_id"] = int(fold_idx)
        hist_df["cv_split_seed"] = int(split_seed)
        history_frames.append(hist_df)

        fold_summary = dict(out["summary"])
        fold_summary.update({
            "fold_id": int(fold_idx),
            "train_size": int(len(fold_train_df)),
            "fold_validation_size": int(len(fold_val_df)),
            "fold_macro_f1": float(fold_out["metrics"]["macro_f1"]),
            "fold_accuracy": float(fold_out["metrics"]["accuracy"]),
            "fold_log_loss": float(fold_out["metrics"]["log_loss"]),
            "checkpoint_dir": str(out["ckpt_dir"]),
            "best_checkpoint_path": str(out["best_checkpoint_path"]),
        })
        fold_summaries.append(fold_summary)

        fold_metric_rows.append({
            "model_kind": model_kind,
            "seed": int(seed),
            "run_name": fold_run_name,
            "fold_id": int(fold_idx),
            "best_epoch": int(out["best_epoch"]),
            "best_val_macro_f1": float(out["best_val_macro_f1"]),
            "oof_accuracy": float(fold_out["metrics"]["accuracy"]),
            "oof_macro_f1": float(fold_out["metrics"]["macro_f1"]),
            "oof_weighted_f1": float(fold_out["metrics"]["weighted_f1"]),
            "oof_balanced_accuracy": float(fold_out["metrics"]["balanced_accuracy"]),
            "oof_log_loss": float(fold_out["metrics"]["log_loss"]),
            "oof_brier_score": float(fold_out["metrics"]["brier_score"]),
            "oof_ece": float(fold_out["metrics"]["ece"]),
            "oof_mce": float(fold_out["metrics"]["mce"]),
            "epochs_ran": int(out["history_df"]["epoch"].max()),
            "mean_epoch_seconds": float(out["history_df"]["epoch_seconds"].mean()) if "epoch_seconds" in out["history_df"].columns else np.nan,
            "peak_memory_mb": float(out["history_df"]["peak_memory_mb"].max()) if "peak_memory_mb" in out["history_df"].columns else np.nan,
            "total_params": fold_summary.get("total_params", np.nan),
            "trainable_params": fold_summary.get("trainable_params", np.nan),
        })

        if save_fold_predictions:
            fold_dir = DIRS["predictions"] / model_kind / f"{run_name_prefix}_fold{fold_idx:02d}"
            save_prediction_bundle(fold_dir, "oof", fold_out)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    oof_output = _build_oof_output(pool_df_, seed, pred_frames, logits_frames, probs_frames, features_frames if len(features_frames) else None)
    fold_metrics_df = pd.DataFrame(fold_metric_rows)
    fold_summary_df = pd.DataFrame(fold_summaries)
    history_df = _aggregate_fold_histories(history_frames, fold_summaries)

    final_epochs = derive_final_epoch_from_folds(fold_summaries, rule=FINAL_EPOCH_RULE)

    return {
        "model_kind": model_kind,
        "seed": int(seed),
        "cfg": cfg,
        "oof_output": oof_output,
        "fold_metrics_df": fold_metrics_df,
        "fold_summary_df": fold_summary_df,
        "history_df": history_df,
        "fold_histories": history_frames,
        "fold_summaries": fold_summaries,
        "best_epoch": int(final_epochs),
        "best_val_macro_f1": float(oof_output["metrics"]["macro_f1"]),
        "selection_metric_name": "pooled_oof_macro_f1",
        "selection_metric_value": float(oof_output["metrics"]["macro_f1"]),
        "cv_n_splits": int(n_splits),
        "cv_split_seed": int(split_seed),
        "final_epoch_rule": str(FINAL_EPOCH_RULE),
        "derived_final_epochs": int(final_epochs),
        "summary": {
            "model_kind": model_kind,
            "seed": int(seed),
            "run_name": run_name_prefix,
            "best_epoch": int(final_epochs),
            "best_val_macro_f1": float(oof_output["metrics"]["macro_f1"]),
            "epochs_ran": int(final_epochs),
            "mean_epoch_seconds": float(history_df["epoch_seconds_mean"].mean()) if "epoch_seconds_mean" in history_df.columns else np.nan,
            "peak_memory_mb": float(history_df["peak_memory_mb_mean"].max()) if "peak_memory_mb_mean" in history_df.columns else np.nan,
            "oof_macro_f1": float(oof_output["metrics"]["macro_f1"]),
            "oof_log_loss": float(oof_output["metrics"]["log_loss"]),
            "selection_metric_name": "pooled_oof_macro_f1",
            "selection_metric_value": float(oof_output["metrics"]["macro_f1"]),
            "total_params": float(fold_summary_df["total_params"].iloc[0]) if "total_params" in fold_summary_df.columns and len(fold_summary_df) else np.nan,
            "trainable_params": float(fold_summary_df["trainable_params"].iloc[0]) if "trainable_params" in fold_summary_df.columns and len(fold_summary_df) else np.nan,
        },
    }

def train_full_pool_fixed_epochs(
    model_kind,
    cfg,
    pool_df_,
    seed,
    final_epochs,
    run_name,
):
    pool_df_ = _ensure_cv_pool_df(pool_df_)
    seed_everything(seed)

    train_tf = build_train_transform(cfg["augmentation_strength"])
    train_loader = make_loader(pool_df_, train_tf, int(cfg["batch_size"]), shuffle=True, seed=seed)

    model = build_model_and_strategy(model_kind, cfg).to(DEVICE)
    total_params = int(sum(p.numel() for p in model.parameters()))
    trainable_params = int(sum(p.numel() for p in model.parameters() if p.requires_grad))

    criterion = loss_fn_builder(label_smoothing=cfg.get("label_smoothing", 0.0))
    optimizer, _ = make_optimizer_and_scheduler(
        model,
        lr_head=cfg["head_lr"],
        lr_backbone=cfg.get("backbone_lr", None),
        weight_decay=cfg["weight_decay"],
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    ckpt_dir = DIRS["checkpoints"] / model_kind / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    history_rows = []
    for epoch in range(1, int(final_epochs) + 1):
        if torch.cuda.is_available():
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass

        t0 = time.perf_counter()
        train_loss, grad_norm = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            grad_accum_steps=GRAD_ACCUM_STEPS,
        )
        epoch_seconds = float(time.perf_counter() - t0)
        peak_memory_mb = float(torch.cuda.max_memory_allocated() / 1024**2) if torch.cuda.is_available() else np.nan

        row = {
            "run_name": run_name,
            "model_kind": model_kind,
            "seed": int(seed),
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "grad_norm": float(grad_norm) if grad_norm is not None else np.nan,
            "lr_group_0": optimizer.param_groups[0]["lr"],
            "epoch_seconds": epoch_seconds,
            "peak_memory_mb": peak_memory_mb,
            "final_epoch_rule": str(FINAL_EPOCH_RULE),
            "derived_final_epochs": int(final_epochs),
            "is_final_epoch": bool(epoch == int(final_epochs)),
            "total_params": total_params,
            "trainable_params": trainable_params,
        }
        if len(optimizer.param_groups) > 1:
            row["lr_group_1"] = optimizer.param_groups[1]["lr"]
        history_rows.append(row)

    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(ckpt_dir / "full_pool_training_history.csv", index=False)

    checkpoint_path = ckpt_dir / "full_pool_checkpoint.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "cfg": cfg,
            "seed": int(seed),
            "model_kind": model_kind,
            "derived_final_epochs": int(final_epochs),
            "label_map": label_map,
            "total_params": total_params,
            "trainable_params": trainable_params,
        },
        checkpoint_path,
    )

    summary_payload = {
        "model_kind": model_kind,
        "run_name": run_name,
        "seed": int(seed),
        "best_epoch": int(final_epochs),
        "best_val_macro_f1": np.nan,
        "epochs_ran": int(final_epochs),
        "mean_epoch_seconds": float(history_df["epoch_seconds"].mean()) if "epoch_seconds" in history_df.columns else np.nan,
        "peak_memory_mb": float(history_df["peak_memory_mb"].max()) if "peak_memory_mb" in history_df.columns else np.nan,
        "total_params": total_params,
        "trainable_params": trainable_params,
        "checkpoint_dir": str(ckpt_dir),
        "best_checkpoint_path": str(checkpoint_path),
    }
    with open(ckpt_dir / "run_summary.json", "w", encoding="utf-8") as f:
        json.dump(_json_safe_obj(summary_payload), f, indent=2)

    return {
        "model": model,
        "history_df": history_df,
        "ckpt_dir": ckpt_dir,
        "best_checkpoint_path": checkpoint_path,
        "summary": summary_payload,
        "best_epoch": int(final_epochs),
        "best_val_macro_f1": np.nan,
    }
